# PneumoScan --- EDA & Preprocessing

This notebook covers the exploratory data analysis and preprocessing pipeline for the **PneumoScan** project --- a multi-class chest X-ray classification system that distinguishes between **Normal**, **Bacterial Pneumonia**, and **Viral Pneumonia**.

### Objectives

1. **Environment setup** --- install dependencies, configure paths, mount Google Drive (if on Colab)
2. **Dataset reorganization** --- convert the original 2-class (Normal / Pneumonia) layout into a 3-class (Normal / Bacteria / Virus) structure
3. **Merge tiny validation set** --- fold the original 24-image val split into training data
4. **Dataset statistics** --- compute and display per-class counts across splits
5. **Class distribution visualization** --- bar charts showing class balance
6. **Sample image visualization** --- grid of representative images per class
7. **Class weight computation** --- handle class imbalance for training
8. **Image size distribution analysis** --- inspect raw image dimensions before resizing

---
## 1. Environment Setup

Install required packages (if not already present) and optionally mount Google Drive for Colab users. We add the project `src/` directory to `sys.path` so that our custom modules can be imported directly.

In [18]:
# === PneumoScan Setup ===
import os, sys, subprocess

# Detect environment and configure paths
try:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = '/content/PneumoScan'
    if not os.path.exists(REPO_DIR):
        subprocess.run(['git', 'clone', 'https://github.com/muhammadrakib2299/PneumoScan.git', REPO_DIR], check=True)
    else:
        subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

    SRC_DIR = os.path.join(REPO_DIR, 'src')
    IN_COLAB = True
    print(f"Running on Google Colab | src: {SRC_DIR}")
except ImportError:
    SRC_DIR = os.path.join(os.path.dirname(os.getcwd()), 'src')
    IN_COLAB = False
    print(f"Running locally | src: {SRC_DIR}")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

# === Imports ===
import config
import preprocessing
import data_loader
import utils

import cv2
import numpy as np
import matplotlib.pyplot as plt

# Configure paths for Colab (dataset on Google Drive)
if IN_COLAB:
    config.setup_colab()

config.ensure_dirs()

print(f"Project root: {config.BASE_DIR}")
print(f"Dataset dir:  {config.RAW_DATA_DIR}")
print(f"Models dir:   {config.MODELS_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running on Google Colab | src: /content/PneumoScan/src
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted.
Repo updated at: /content/PneumoScan
Dataset path: /content/drive/MyDrive/chest_xray
Models save to: /content/drive/MyDrive/PneumoScan_checkpoints
Project dir: /content/PneumoScan
Setup complete!
Project root: /content/PneumoScan
Dataset dir:  /content/drive/MyDrive/chest_xray
Models dir:   /content/drive/MyDrive/PneumoScan_checkpoints


---
## 2. Dataset Reorganization

The original Kaggle *Chest X-Ray Images (Pneumonia)* dataset uses a binary structure:

```
train/
  NORMAL/
  PNEUMONIA/       <-- contains both bacteria_*.jpeg and virus_*.jpeg
```

We reorganize it into three classes based on filename prefixes:

```
train/
  NORMAL/
  BACTERIA/        <-- files containing 'bacteria' in the name
  VIRUS/           <-- files containing 'virus' in the name
```

This uses `shutil.copy2` (not move), so the original PNEUMONIA folder is preserved.

In [19]:
# Reorganize PNEUMONIA into BACTERIA / VIRUS across all splits
reorg_stats = preprocessing.reorganize_dataset()
reorg_stats


TRAIN split:
  NORMAL: 0 (0.0%)
  BACTERIA: 0 (0.0%)
  VIRUS: 0 (0.0%)
  Total: 0

VAL split:
  NORMAL: 0 (0.0%)
  BACTERIA: 0 (0.0%)
  VIRUS: 0 (0.0%)
  Total: 0

TEST split:
  NORMAL: 0 (0.0%)
  BACTERIA: 0 (0.0%)
  VIRUS: 0 (0.0%)
  Total: 0


{'train': {'NORMAL': 0, 'BACTERIA': 0, 'VIRUS': 0},
 'val': {'NORMAL': 0, 'BACTERIA': 0, 'VIRUS': 0},
 'test': {'NORMAL': 0, 'BACTERIA': 0, 'VIRUS': 0}}

---
## 3. Merge Validation Set into Training

The original dataset has only **24 validation images** --- far too few for meaningful validation. We merge them into the training set and will instead use a stratified 80/20 train-val split (handled in `data_loader.load_train_val_split()`).

In [20]:
moved = preprocessing.merge_val_into_train()
print(f"Total images moved from val -> train: {moved}")


Merged 0 validation images into training set.
Total images moved from val -> train: 0


---
## 4. Dataset Statistics

After reorganization and merging, let's inspect the final class counts across all splits.

In [21]:
stats = preprocessing.get_dataset_stats()

for split, counts in stats.items():
    total = sum(counts.values())
    print(f"\n{split.upper()} ({total} total):")
    for cls, count in counts.items():
        pct = (count / total * 100) if total > 0 else 0
        print(f"  {cls:10s}: {count:5d}  ({pct:.1f}%)")


TRAIN (0 total):
  NORMAL    :     0  (0.0%)
  BACTERIA  :     0  (0.0%)
  VIRUS     :     0  (0.0%)

VAL (0 total):
  NORMAL    :     0  (0.0%)
  BACTERIA  :     0  (0.0%)
  VIRUS     :     0  (0.0%)

TEST (0 total):
  NORMAL    :     0  (0.0%)
  BACTERIA  :     0  (0.0%)
  VIRUS     :     0  (0.0%)


---
## 5. Class Distribution Visualization

A bar chart makes the class imbalance immediately obvious. Bacterial pneumonia typically dominates the dataset, which motivates our use of class weights during training.

In [22]:
# Plot class distribution for the training set
train_counts = stats.get("train", {})
utils.plot_class_distribution(
    train_counts,
    title="Training Set --- Class Distribution",
    save_path=os.path.join(config.EDA_FIGURES_DIR, "class_distribution_train.png"),
)

/content/PneumoScan/src/utils.py:176: UserWarning: Tight layout not applied. The bottom and top margins cannot be made large enough to accommodate all Axes decorations.
  plt.tight_layout()


Saved: /content/PneumoScan/outputs/figures/eda/class_distribution_train.png


In [23]:
# Plot class distribution for the test set
test_counts = stats.get("test", {})
utils.plot_class_distribution(
    test_counts,
    title="Test Set --- Class Distribution",
    save_path=os.path.join(config.EDA_FIGURES_DIR, "class_distribution_test.png"),
)

Saved: /content/PneumoScan/outputs/figures/eda/class_distribution_test.png


---
## 6. Sample Image Visualization

We load the training dataset **without augmentation** (to see the raw images) and display a grid of sample images from each class. This helps verify that the reorganization worked correctly and gives a qualitative feel for inter-class differences.

In [24]:
# Load training dataset without augmentation for visualization
viz_ds = data_loader.load_train_dataset(augment=False)

# Plot 4 sample images per class
utils.plot_sample_images(
    viz_ds,
    n_per_class=4,
    save_path=os.path.join(config.EDA_FIGURES_DIR, "sample_images.png"),
)

ValueError: The `class_names` passed did not match the names of the subdirectories of the target directory. Expected: ['BACTERIA', 'VIRUS'] (or a subset of it), but received: class_names=['NORMAL', 'BACTERIA', 'VIRUS']

---
## 7. Class Weights

Since the dataset is imbalanced (more bacterial pneumonia images than normal or viral), we compute inverse-frequency class weights. These will be passed to `model.fit()` via the `class_weight` parameter so the loss function penalizes misclassifications of minority classes more heavily.

$$w_c = \frac{N_{total}}{K \times N_c}$$

where $K$ is the number of classes and $N_c$ is the count for class $c$.

In [ ]:
class_weights = data_loader.compute_class_weights()
print(f"\nClass weights dict: {class_weights}")

---
## 8. Image Size Distribution Analysis

Before the data pipeline resizes everything to 224x224, let's examine the raw image dimensions. This helps us understand:

- Whether significant up-scaling or down-scaling occurs
- The aspect ratio distribution (are images roughly square?)
- Whether any anomalously small or large images exist

In [ ]:
# Collect raw image dimensions from the training directory
widths = []
heights = []

train_dir = config.TRAIN_DIR
for class_name in config.CLASS_NAMES:
    class_dir = os.path.join(train_dir, class_name)
    if not os.path.exists(class_dir):
        continue
    for root, dirs, files in os.walk(class_dir):
        for fname in files:
            if not fname.lower().endswith((".jpeg", ".jpg", ".png")):
                continue
            fpath = os.path.join(root, fname)
            img = cv2.imread(fpath)
            if img is not None:
                h, w = img.shape[:2]
                heights.append(h)
                widths.append(w)

widths = np.array(widths)
heights = np.array(heights)

print(f"Analyzed {len(widths)} images")
print(f"Width  --- min: {widths.min()}, max: {widths.max()}, mean: {widths.mean():.0f}, median: {np.median(widths):.0f}")
print(f"Height --- min: {heights.min()}, max: {heights.max()}, mean: {heights.mean():.0f}, median: {np.median(heights):.0f}")

In [ ]:
# Plot histograms of width and height distributions
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Width distribution
axes[0].hist(widths, bins=50, color="#2196F3", edgecolor="black", alpha=0.8)
axes[0].axvline(x=224, color="red", linestyle="--", linewidth=2, label="Target (224)")
axes[0].set_xlabel("Width (pixels)")
axes[0].set_ylabel("Count")
axes[0].set_title("Image Width Distribution")
axes[0].legend()
axes[0].grid(axis="y", alpha=0.3)

# Height distribution
axes[1].hist(heights, bins=50, color="#4CAF50", edgecolor="black", alpha=0.8)
axes[1].axvline(x=224, color="red", linestyle="--", linewidth=2, label="Target (224)")
axes[1].set_xlabel("Height (pixels)")
axes[1].set_ylabel("Count")
axes[1].set_title("Image Height Distribution")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

# Aspect ratio distribution
aspect_ratios = widths / heights
axes[2].hist(aspect_ratios, bins=50, color="#FF9800", edgecolor="black", alpha=0.8)
axes[2].axvline(x=1.0, color="red", linestyle="--", linewidth=2, label="Square (1:1)")
axes[2].set_xlabel("Aspect Ratio (W/H)")
axes[2].set_ylabel("Count")
axes[2].set_title("Aspect Ratio Distribution")
axes[2].legend()
axes[2].grid(axis="y", alpha=0.3)

fig.suptitle("Raw Image Size Distribution (Training Set)", fontsize=14, fontweight="bold")
plt.tight_layout()

save_path = os.path.join(config.EDA_FIGURES_DIR, "image_size_distribution.png")
utils.save_figure(fig, save_path)
plt.show()

---
## Summary

In this notebook we completed the following:

| Step | Description | Status |
|------|-------------|--------|
| 1 | Environment setup and imports | Done |
| 2 | Reorganized PNEUMONIA into BACTERIA / VIRUS | Done |
| 3 | Merged tiny val set into training data | Done |
| 4 | Computed per-split dataset statistics | Done |
| 5 | Visualized class distributions | Done |
| 6 | Displayed sample images per class | Done |
| 7 | Computed class weights for imbalance handling | Done |
| 8 | Analyzed raw image size distributions | Done |

### Key Takeaways

- The dataset is **imbalanced** --- bacterial pneumonia has the most samples, viral pneumonia has the fewest. Class weights will be essential during training.
- Most raw images are larger than 224x224, meaning our pipeline **downscales** rather than upscales (good for preserving detail).
- Aspect ratios are generally close to 1:1, so resizing to a square introduces minimal distortion.

**Next:** Proceed to `02_baseline_custom_cnn.ipynb` to train the baseline Custom CNN model.